In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import confusion_matrix
import seaborn as sns

Modell laden 

In [2]:
model = AutoModelForSequenceClassification.from_pretrained("manifesto-project/manifestoberta-xlm-roberta-56policy-topics-context-2024-1-1", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-large")

Wahlomat klassifizieren

In [ ]:
# Wahlomat-Thesen zu Manifesto Kategorien mappen, z.B. These 1 wird zu Welfare State Expansion gemappt 
# Wahlomat-Thesen laden
df_questions = pd.read_csv("wahlomaten_thesen_positionen.csv")

# Liste für Ergebnisse
results = []

for index, row in df_questions.iterrows():
    sentence = row["These"]
    context = sentence  # Nutzen der These selbst, als Versuch, sonst wohl text aus txt datei nehmen?

    # Tokenisierung, Max len angepasst, muss im finetuning geändert werden 
    inputs = tokenizer(sentence, context, return_tensors="pt", max_length=512, padding="max_length", truncation=True)

    # Modellvorhersage
    logits = model(**inputs).logits
    probabilities = torch.softmax(logits, dim=1).tolist()[0]
    
    # Mapping der ID zur Policy-Kategorie
    probabilities = {model.config.id2label[index]: round(probability * 100, 2) for index, probability in enumerate(probabilities)}
    probabilities = dict(sorted(probabilities.items(), key=lambda item: item[1], reverse=True))
    
    # Beste Kategorie auswählen, Alternativ könnten hier auch alle Wahrscheinlichkeiten zum Vergleich genommen werden
    predicted_class = model.config.id2label[logits.argmax().item()]

    results.append({"These": sentence, "Best_Policy_Category": predicted_class, "Full_Probabilities": probabilities})

# Ergebnisse als DataFrame speichern
df_results = pd.DataFrame(results)
df_results.to_csv("wahlomat_policy_predictions.csv", index=False)

print("Wahlomat-Kategorien gespeichert in 'wahlomat_policy_predictions.csv'")

Parteien klassifizieren

In [ ]:
df = pd.read_parquet("all_parties_text_combined_prep.parquet")

# Duplikate aus Thesen entfernen
df_filtered = df.dropna(subset=["These"]).drop_duplicates(subset=["Party", "These"])

all_results = []

# Klassifizierung der Thesen
for index, row in df_filtered.iterrows():
    party = row["Party"]
    thesis = row["These"]
    context = row["Text"]  # Jede These wird im Kontext ihres zugehörigen Textes klassifiziert
    #print("Started: ", index)

    # Vorhersage
    inputs = tokenizer(thesis, context, return_tensors="pt", max_length=512, padding="max_length", truncation=True)
    logits = model(**inputs).logits
    #print("Predicted: ", index)

    # Wahrscheinlichkeit berechnen 
    probabilities = torch.softmax(logits, dim=1).tolist()[0]
    predicted_class = model.config.id2label[logits.argmax().item()]
    #print("probability: ", index)

    all_results.append({
        "Partei": party,
        "These": thesis,
        "Text": context,
        "Policy_Category": predicted_class
    })

#print("End of for loop")
# Ergebnisse speichern
df_results = pd.DataFrame(all_results)

df_results.to_csv("classified_text_results_all_parties.csv", index=False)
print("Partei-Kategorien gespeichert in 'classified_text_results_all_parties.csv'")


Parteien und Wahlomat vergleichen

In [ ]:
# Aus Klassifizierten Wahlomat und Partei Texten übereinstimmung analysieren
# Wahlomat-Daten
df_wahlomat = pd.read_csv("wahlomat_policy_predictions.csv")
df_wahlomat = df_wahlomat[["These", "Best_Policy_Category"]]

# Partei-Daten 
df_parties = pd.read_csv("classified_text_results_all_parties.csv")  # Enthält Positionen aller Parteien

# Mapping der Klassen auf Basis der Domainen 
def determine_position(wahlomat_category, party_categories):
    """
    Bestimmt die Haltung einer Partei basierend auf der Policy-Kategorie der Wahlomat-These.
    """
    opposite_policies = {
        "504 - Welfare State Expansion": "505 - Welfare State Limitation",
        "505 - Welfare State Limitation": "504 - Welfare State Expansion",
        "701 - Labour Groups: Positive": "702 - Labour Groups: Negative",
        "702 - Labour Groups: Negative": "701 - Labour Groups: Positive",
        "406 - Protectionism: Positive": "407 - Protectionism: Negative",
        "407 - Protectionism: Negative": "406 - Protectionism: Positive",
        "506 - Education Expansion": "507 - Education Limitation",
        "507 - Education Limitation": "506 - Education Expansion",
        "401 - Free Market Economy": "412 - Controlled Economy",
        "412 - Controlled Economy": "401 - Free Market Economy",
        "101 - Foreign Special Relationships: Positive": "102 - Foreign Special Relationships: Negative",
        "102 - Foreign Special Relationships: Negative": "101 - Foreign Special Relationships: Positive",
        "103 - Anti-Imperialism": "108 - European Community/Union: Positive",
        "104 - Military: Positive": "105 - Military: Negative",
        "105 - Military: Negative": "104 - Military: Positive",
        "106 - Peace": "104 - Military: Positive",
        "107 - Internationalism: Positive": "109 - Internationalism: Negative",
        "109 - Internationalism: Negative": "107 - Internationalism: Positive",
        "108 - European Community/Union: Positive": "110 - European Community/Union: Negative",
        "110 - European Community/Union: Negative": "108 - European Community/Union: Positive",
        "201 - Freedom and Human Rights": "605 - Law and Order: Positive",
        "202 - Democracy": "305 - Political Authority",
        "203 - Constitutionalism: Positive": "204 - Constitutionalism: Negative",
        "204 - Constitutionalism: Negative": "203 - Constitutionalism: Positive",
        "301 - Federalism": "302 - Centralisation",
        "302 - Centralisation": "301 - Federalism",
        "303 - Governmental and Administrative Efficiency": "304 - Political Corruption",
        "304 - Political Corruption": "303 - Governmental and Administrative Efficiency",
        "305 - Political Authority": "202 - Democracy",
        "402 - Incentives: Positive": "409 - Keynesian Demand Management",
        "409 - Keynesian Demand Management": "402 - Incentives: Positive",
        "403 - Market Regulation": "404 - Economic Planning",
        "404 - Economic Planning": "403 - Market Regulation",
        "405 - Corporatism/ Mixed Economy": "401 - Free Market Economy",
        "410 - Economic Growth: Positive": "416 - Anti-Growth Economy: Positive",
        "416 - Anti-Growth Economy: Positive": "410 - Economic Growth: Positive",
        "501 - Environmental Protection: Positive": "502 - Culture: Positive",
        "601 - National Way of Life: Positive": "602 - National Way of Life: Negative",
        "602 - National Way of Life: Negative": "601 - National Way of Life: Positive",
        "603 - Traditional Morality: Positive": "604 - Traditional Morality: Negative",
        "604 - Traditional Morality: Negative": "603 - Traditional Morality: Positive",
        "605 - Law and Order: Positive": "201 - Freedom and Human Rights",
        "606 - Civic Mindedness: Positive": "604 - Traditional Morality: Negative",
        "607 - Multiculturalism: Positive": "608 - Multiculturalism: Negative",
        "608 - Multiculturalism: Negative": "607 - Multiculturalism: Positive",
        "704 - Middle Class and Professional Groups": "705 - Underprivileged Minority Groups",
        "705 - Underprivileged Minority Groups": "704 - Middle Class and Professional Groups"
    }


    if wahlomat_category in party_categories:
        return "stimme zu"
    elif wahlomat_category in opposite_policies and opposite_policies[wahlomat_category] in party_categories:
        return "stimme nicht zu"
    else:
        return "neutral"

party_names = df_parties["Partei"].unique()  # Alle Parteien im Datensatz

# Neuen DataFrame vorbereiten
df_wahlomat_results = df_wahlomat.copy()

for party in party_names:
    # Kategorien für die aktuelle Partei
    party_policy_set = set(df_parties[df_parties["Partei"] == party]["Policy_Category"].unique())

    # Partei-Position für jede Wahlomat-These
    df_wahlomat_results[f"{party}_Model_Prediction"] = df_wahlomat_results["Best_Policy_Category"].apply(
        lambda cat: determine_position(cat, party_policy_set)
    )


df_wahlomat_results.to_csv("all_parties_wahlomat_policy_matching.csv", index=False)

Ergebnisse visualisieren 

In [ ]:
# Visualisieren der Ergebnisse

#  Wahlomat-Daten (Original // True Werte)
df_wahlomat_true = pd.read_csv("wahlomaten_thesen_positionen.csv")  # Enthält tatsächliche Wahlomat-Antworten
df_wahlomat_true = df_wahlomat_true.rename(columns={"CDU / CSU": "CDU"})  
df_wahlomat_true = df_wahlomat_true[["These", "CDU", "GRÜNE", "SPD", "AfD", "Die Linke", "FDP"]]  # Nur relevante Spalten behalten

# Predictions für Parteien --> These entspricht hier der Wahlomat These
df_wahlomat_pred = pd.read_csv("all_parties_wahlomat_policy_matching.csv")  # Modell-Vorhersagen der Parteien


df_comparison = df_wahlomat_true.merge(df_wahlomat_pred, on="These", how="left")

# Übereinstimmungen prüfen 
parties = ["CDU", "GRÜNE", "SPD", "AfD", "Die Linke", "FDP"]

party_model_mapping = {
    "CDU": "cdu_Model_Prediction",
    "GRÜNE": "gruene_Model_Prediction",
    "SPD": "spd_Model_Prediction",
    "AfD": "afd_Model_Prediction",
    "Die Linke": "linke_Model_Prediction",
    "FDP": "fdp_Model_Prediction"
}

# Übereinstimmungen prüfen 
results = {}
for party, model_col in party_model_mapping.items():
    if model_col in df_comparison.columns:  # Sicherstellen, dass die Spalte existiert
        df_comparison[f"Abweichung_{party}"] = df_comparison.apply(
            lambda row: "Korrekt" if row[party] == row[model_col] else "Abweichung", axis=1
        )
        results[party] = df_comparison[f"Abweichung_{party}"].value_counts()


# Plot 
plt.figure(figsize=(12, 6))
bar_width = 0.25
index = range(len(parties)) 

# Balkendiagramm für jede Partei
correct_counts = [results[party].get("Korrekt", 0) for party in parties]
incorrect_counts = [results[party].get("Abweichung", 0) for party in parties]

plt.bar(index, correct_counts, bar_width, label="Korrekt", color="green", edgecolor="black")
plt.bar(index, incorrect_counts, bar_width, bottom=correct_counts, label="Abweichung", color="red", edgecolor="black")

plt.title("Vergleich der Modellvorhersagen mit den echten Wahlomat-Positionen")
plt.xlabel("Parteien")
plt.ylabel("Anzahl der Thesen")
plt.xticks(index, parties, rotation=30)
plt.legend(loc="upper right")
plt.show()

# Ergebnisse speichern
df_comparison.to_csv("all_parties_wahlomat_vergleich_plot.csv", index=False)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))  # 2 Reihen, 3 Spalten für Parteien

for i, (party, model_col) in enumerate(party_model_mapping.items()):
    if model_col in df_comparison.columns:  # Sicherstellen, dass die Spalte existiert
        # **Erstellen der Confusion Matrix**
        cm = confusion_matrix(df_comparison[party], df_comparison[model_col], labels=["stimme zu", "neutral", "stimme nicht zu"])
        
        # **Visualisierung der Confusion Matrix mit Seaborn**
        ax = axes[i // 3, i % 3]  # Positionierung im Grid
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["stimme zu", "neutral", "stimme nicht zu"], 
                    yticklabels=["stimme zu", "neutral", "stimme nicht zu"], ax=ax)
        
        ax.set_title(f"Confusion Matrix - {party}")
        ax.set_xlabel("Vorhergesagt")
        ax.set_ylabel("Tatsächlich")

plt.tight_layout()